# Exportación y análisis de traslados GRD 2024

Notebook final del flujo: usa los CSV procesados para revisar derivaciones, gravedad y dejar insumos listos para el dashboard.

La siguiente celda carga traslados, motivos y hospitales para preparar el análisis de derivaciones.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

ROOT_DIR = Path.cwd().resolve()
for candidate in [ROOT_DIR, *ROOT_DIR.parents]:
    if (candidate / 'app').exists() and (candidate / 'data').exists():
        ROOT_DIR = candidate
        break

sys.path.insert(0, str(ROOT_DIR / 'scripts'))
import grd_common as grd

print(f'Raíz del proyecto: {ROOT_DIR}')

Raíz del proyecto: C:\Users\Bato\Desktop\Infe\Proyecto-ADIE


La siguiente celda resume la distribución de motivos de traslado para identificar las causas más frecuentes.

In [2]:
processed_dir = ROOT_DIR / 'data' / 'processed'
traslados = pd.read_csv(processed_dir / 'traslados.csv')
motivo_traslado = pd.read_csv(processed_dir / 'motivo_traslado.csv')
hospitales = pd.read_csv(processed_dir / 'hospitales.csv')

print('Traslados cargados:', len(traslados))
print('Motivos cargados:', len(motivo_traslado))
print('Hospitales cargados:', len(hospitales))

Traslados cargados: 196
Motivos cargados: 215851
Hospitales cargados: 70


Aquí se visualiza el flujo de traslados sobre el mapa para ubicar los principales orígenes y destinos.

In [3]:
top_routes = traslados.sort_values('cantidad', ascending=False).head(15).copy()

fig_routes = px.bar(
    top_routes,
    x='cantidad',
    y='hospital_destino',
    orientation='h',
    title='Top 15 rutas de traslado por volumen'
)
fig_routes.show()

fig_map = go.Figure()
for _, row in top_routes.iterrows():
    fig_map.add_trace(go.Scattergeo(
        lon=[row['lon_origen'], row['lon_destino']],
        lat=[row['lat_origen'], row['lat_destino']],
        mode='lines',
        line=dict(width=max(1, row['cantidad'] / 120), color='rgba(180, 40, 40, 0.45)'),
        hoverinfo='none',
        showlegend=False
    ))

fig_map.add_trace(go.Scattergeo(
    lon=top_routes['lon_origen'],
    lat=top_routes['lat_origen'],
    text=top_routes['hospital_origen'],
    mode='markers',
    marker=dict(size=6, color='steelblue'),
    name='Origen'
))

fig_map.add_trace(go.Scattergeo(
    lon=top_routes['lon_destino'],
    lat=top_routes['lat_destino'],
    text=top_routes['hospital_destino'],
    mode='markers',
    marker=dict(size=8, color='darkred'),
    name='Destino'
))

fig_map.update_geos(fitbounds='locations', visible=False)
fig_map.update_layout(title='Mapa de las principales derivaciones hospitalarias')
fig_map.show()

La siguiente celda estima la proporción de traslados con severidad alta e incluye su intervalo de confianza al 95%.

In [4]:
resumen_motivo = motivo_traslado.groupby(['severidad', 'mortalidad']).size().reset_index(name='cantidad')
print('Distribución de gravedad y mortalidad en derivaciones:')
display(resumen_motivo.sort_values('cantidad', ascending=False).head(20))

print('Top 10 diagnósticos en derivaciones:')
display(motivo_traslado['diagnostico'].value_counts().rename_axis('diagnostico').reset_index(name='cantidad').head(10))

Distribución de gravedad y mortalidad en derivaciones:


,severidad,mortalidad,cantidad
9,3.0,3.0,65782
1,1.0,1.0,56804
5,2.0,2.0,28880
8,3.0,2.0,23030
4,2.0,1.0,21587
2,1.0,2.0,5835
6,2.0,3.0,5719
0,0.0,0.0,4111
7,3.0,1.0,3758
3,1.0,3.0,339


Top 10 diagnósticos en derivaciones:


,diagnostico,cantidad
0,U07.1,14457
1,I21.4,3764
2,J18.9,3021
3,I25.1,2803
4,S72.00,2678
5,E11.5,2636
6,I50.0,2088
7,N39.0,2053
8,P22.0,2030
9,I21.0,2026


La siguiente celda resume la distribución de severidad y mortalidad en las derivaciones, además de los diagnósticos más frecuentes.

## Inferencia estadistica en traslados

Se incorporan intervalos de confianza y pruebas de hipotesis para severidad y mortalidad en derivaciones.

La siguiente celda construye la tabla de contingencia entre severidad y mortalidad para evaluar su asociación.

In [5]:
from math import sqrt
from statistics import NormalDist


def wilson_ci(successes: float, n: float, confidence: float = 0.95):
    if n <= 0:
        return (float('nan'), float('nan'))
    z = NormalDist().inv_cdf(1 - (1 - confidence) / 2)
    p = successes / n
    denom = 1 + (z**2) / n
    center = (p + (z**2) / (2 * n)) / denom
    margin = (z / denom) * sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))
    return (max(0.0, center - margin), min(1.0, center + margin))

Aquí se aplica la prueba chi-cuadrado para medir si severidad y mortalidad están asociadas.

In [6]:
# IC95 de proporcion de severidad alta (>= 3) en traslados
motivo_tmp = motivo_traslado.copy()
motivo_tmp['severidad'] = pd.to_numeric(motivo_tmp['severidad'], errors='coerce')

n_total = motivo_tmp['severidad'].notna().sum()
n_sev_alta = (motivo_tmp['severidad'] >= 3).sum()

ci_low, ci_high = wilson_ci(n_sev_alta, n_total)
prop = (n_sev_alta / n_total) if n_total > 0 else float('nan')

print('Proporcion de severidad alta en traslados:')
print(f'  Estimacion puntual: {prop*100:.2f}%')
print(f'  IC95: [{ci_low*100:.2f}%, {ci_high*100:.2f}%]')

Proporcion de severidad alta en traslados:
  Estimacion puntual: 42.89%
  IC95: [42.68%, 43.10%]


La siguiente celda compara hospitales destino con mayor volumen y calcula la proporción de casos graves.

In [7]:
# Prueba de independencia: severidad vs mortalidad (chi-cuadrado)
try:
    from scipy.stats import chi2_contingency
except ImportError:
    chi2_contingency = None

chi_data = motivo_traslado.copy()
chi_data['severidad'] = pd.to_numeric(chi_data['severidad'], errors='coerce')
chi_data['mortalidad'] = pd.to_numeric(chi_data['mortalidad'], errors='coerce')
chi_data = chi_data.dropna(subset=['severidad', 'mortalidad']).copy()

if chi2_contingency is None:
    print('scipy no instalado: no se puede ejecutar chi-cuadrado en este notebook.')
else:
    tabla_cont = pd.crosstab(chi_data['severidad'], chi_data['mortalidad'])
    chi2, p_value, dof, expected = chi2_contingency(tabla_cont)
    print('Tabla de contingencia severidad x mortalidad:')
    display(tabla_cont)
    print(f'chi2 = {chi2:.4f}, p-value = {p_value:.6g}, dof = {dof}')

Tabla de contingencia severidad x mortalidad:


mortalidad,0.0,1.0,2.0,3.0
severidad,,,,
0.0,4111,0,0,0
1.0,0,56804,5835,339
2.0,0,21587,28880,5719
3.0,0,3758,23030,65782


chi2 = 377940.7519, p-value = 0, dof = 9


In [8]:
# Comparacion entre hospitales de alto volumen de traslado (severidad alta)
routes_tmp = traslados.copy()

if 'graves' in routes_tmp.columns and 'cantidad' in routes_tmp.columns:
    routes_tmp['graves'] = pd.to_numeric(routes_tmp['graves'], errors='coerce').fillna(0)
    routes_tmp['cantidad'] = pd.to_numeric(routes_tmp['cantidad'], errors='coerce').fillna(0)

    by_hospital = (
        routes_tmp.groupby('hospital_destino', as_index=False)
        .agg(graves=('graves', 'sum'), cantidad=('cantidad', 'sum'))
    )
    by_hospital = by_hospital[by_hospital['cantidad'] >= 50].copy()
    by_hospital['prop_graves'] = by_hospital['graves'] / by_hospital['cantidad']

    top_hosp = by_hospital.sort_values('cantidad', ascending=False).head(8)
    print('Hospitales destino (alto volumen) con proporcion de graves:')
    display(top_hosp[['hospital_destino', 'cantidad', 'graves', 'prop_graves']])
else:
    print('El archivo traslados.csv no trae columna graves; se omite comparacion por hospital.')

Hospitales destino (alto volumen) con proporcion de graves:


,hospital_destino,cantidad,graves,prop_graves
1,Complejo Hospitalario Dr. Sótero del Río (Sant...,514,102,0.198444
23,Hospital Dr. Hernán Henríquez Aravena (Temuco),295,108,0.366102
47,Hospital San Pablo (Coquimbo),251,82,0.326693
11,Hospital Clínico Regional Dr. Guillermo Grant ...,146,89,0.609589
7,Hospital Clínico Herminda Martín (Chillán),126,55,0.436508
41,Hospital San Juan de Dios (La Serena),123,54,0.439024
60,Instituto Nacional de Enfermedades Respiratori...,80,46,0.575000
33,Hospital Presidente Carlos Ibáñez del Campo (L...,71,40,0.563380


La última celda deja listo el resumen de hospitales destino con más derivaciones graves para su interpretación final.